# MinneApple 학습 전 기준 모델 평가

MinneApple로 추가 학습하기 전의 COCO 사전학습 YOLOv8n-seg를 기존 검증 이미지 152장과 사과 정답 6,101개로 평가합니다. 원본 가중치와 학습 결과를 다시 학습하지 않습니다.

아래 셀을 실행하면 전송해 둔 Drive 데이터에서 검증 자료를 꺼내고, 정답의 사과 클래스 번호를 사전학습 모델에 맞춥니다. 원본 파일은 보존합니다. 연결된 GPU가 있으면 FP16으로 평가하고, CPU 런타임에서는 FP32를 사용합니다.

결과는 프로젝트 Drive의 outputs/pretrained_eval_<UTC시각> 아래 metrics.csv, summary.json과 평가 그래프에 기록합니다.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, io, json, math, shutil, subprocess, sys, tempfile, traceback, zipfile
from google.colab import drive

project = Path("/content/drive/MyDrive/apple-harvesting-robot-colab-20261116")
if not project.is_dir():
    drive.mount("/content/drive")
if not project.is_dir():
    raise FileNotFoundError(project)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = project / "outputs" / f"pretrained_eval_{stamp}"
out.mkdir(parents=True, exist_ok=False)
print("결과 저장 위치:", out, flush=True)
record = {"status": "preparing", "weights": "Original COCO yolov8n-seg.pt",
          "split": "MinneApple fixed v1 val", "images": 152, "apple_annotations": 6101}
def save_record():
    tmp = out / "summary.tmp"
    tmp.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(out / "summary.json")
save_record()

try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
    import torch, ultralytics
    from ultralytics import YOLO
    from ultralytics.utils import YAML
    if ultralytics.__version__ != "8.4.215":
        raise RuntimeError(f"Expected Ultralytics 8.4.215, got {ultralytics.__version__}")

    with tempfile.TemporaryDirectory(prefix="minneapple_pretrained_", dir="/content") as temp:
        root = Path(temp)
        archive = root / "source.zip"
        archive_hash = hashlib.sha256()
        with archive.open("wb") as destination:
            for part, size in [(1, 419430400), (2, 419430400), (3, 407453432)]:
                source = project / f"minneapple_colab_v1.zip.{part:03d}"
                if source.stat().st_size != size:
                    raise ValueError(f"Unexpected input size: {source.name}")
                with source.open("rb") as stream:
                    while block := stream.read(4 * 1024 * 1024):
                        archive_hash.update(block)
                        destination.write(block)
        if archive_hash.hexdigest() != "3c27555ed6642010b854db3e6e0ed5f83c8c0454a665ca3df23087cd72d2933e":
            raise ValueError("MinneApple ZIP SHA-256 mismatch")
        record["dataset_archive_sha256"] = archive_hash.hexdigest()
        print("기존 데이터 묶음 검증 완료", flush=True)

        (root / "images/val").mkdir(parents=True)
        (root / "labels/val").mkdir(parents=True)
        with zipfile.ZipFile(archive) as z:
            manifest_text = z.read("splits/manifest.csv").decode("utf-8-sig")
            manifest = list(csv.DictReader(io.StringIO(manifest_text)))
            selected = [row for row in manifest if row["split"] == "val"]
            if len(selected) != 152 or len({row["filename"] for row in selected}) != 152:
                raise ValueError("Fixed validation split mismatch")
            weights = root / "yolov8n-seg.pt"
            weights.write_bytes(z.read("weights/yolov8n-seg.pt"))
            record["weights_sha256"] = hashlib.sha256(weights.read_bytes()).hexdigest()
            model = YOLO(str(weights))
            names = {int(key): value for key, value in model.names.items()}
            apple_ids = [key for key, value in names.items() if value == "apple"]
            if len(apple_ids) != 1 or len(names) != 80:
                raise ValueError(f"Expected original 80-class weights; apple IDs: {apple_ids}")
            apple_id = apple_ids[0]

            annotations = 0
            for row in selected:
                filename = row["filename"]
                stem = Path(filename).stem
                image_rel = f"images/val/{filename}"
                raw_image = z.read(image_rel)
                if hashlib.sha256(raw_image).hexdigest() != row["image_sha256"]:
                    raise ValueError(f"Unexpected validation image: {filename}")
                (root / image_rel).write_bytes(raw_image)
                label_rel = f"labels/val/{stem}.txt"
                remapped = []
                for line in z.read(label_rel).decode("utf-8-sig").splitlines():
                    values = line.split()
                    if not values:
                        continue
                    if values[0] != "0" or len(values) < 7 or len(values) % 2 != 1:
                        raise ValueError(f"Unexpected polygon label: {label_rel}")
                    remapped.append(" ".join([str(apple_id), *values[1:]]))
                if len(remapped) != int(row["instance_id_count"]):
                    raise ValueError(f"Annotation count changed: {label_rel}")
                (root / label_rel).write_text("\n".join(remapped) + "\n", encoding="utf-8")
                annotations += len(remapped)
        if annotations != 6101:
            raise ValueError(f"Expected exactly 6101 apples, got {annotations}")

        dataset = root / "eval.yaml"
        YAML.save(str(dataset), {"path": str(root), "train": "images/val", "val": "images/val", "names": names})
        # Both paths satisfy the validator's schema. Only split="val" is executed.
        cuda = torch.cuda.is_available()
        options = dict(split="val", imgsz=640, batch=2, workers=0, cache=False, rect=True,
                       conf=0.001, iou=0.7, max_det=300, classes=[apple_id], single_cls=False,
                       augment=False, mask_ratio=1, overlap_mask=True, plots=True,
                       device=0 if cuda else "cpu", quantize=16 if cuda else None)
        record.update(status="evaluating", eval_args=options, ultralytics=ultralytics.__version__,
                      torch=str(torch.__version__), gpu=torch.cuda.get_device_name(0) if cuda else None)
        save_record()
        print(f"학습 전 COCO 모델 검증 시작: 152장 / apple ID={apple_id}", flush=True)
        metrics = model.val(data=str(dataset), project=str(root), name="plots", **options)

        if metrics.nt_per_class is None or int(metrics.nt_per_class.sum()) != 6101:
            raise ValueError("Validator ground truth count is not 6101")
        fields = ("precision", "recall", "mAP50", "mAP50-95")
        report = []
        for label, suffix in (("box", "B"), ("mask", "M")):
            scores = {field: float(metrics.results_dict[f"metrics/{field}({suffix})"]) for field in fields}
            if not all(math.isfinite(value) and 0.0 <= value <= 1.0 for value in scores.values()):
                raise ValueError(f"Invalid validation metrics: {scores}")
            report.append({"model": "COCO pretrained before MinneApple", "task": label, **scores})
        with (out / "metrics.csv").open("w", encoding="utf-8-sig", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=["model", "task", *fields])
            writer.writeheader()
            writer.writerows(report)
        for figure in (root / "plots").glob("*"):
            if figure.is_file():
                shutil.copy2(figure, out / figure.name)
        record.update(status="completed", validator_apple_count=int(metrics.nt_per_class.sum()), metrics=report)
        save_record()
        print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)
        print("평가 완료. Drive 저장 위치:", out, flush=True)
except Exception as exc:
    record.update(status="failed", error=f"{type(exc).__name__}: {exc}")
    save_record()
    (out / "error.txt").write_text(traceback.format_exc(), encoding="utf-8")
    raise
